# Cross-Session Memory

> *Saving agent state across independent sessions so returning users get continuity, not a blank slate.*

Think of a coworker who loses their memory overnight. Each morning, you'd re-explain your project, your preferences, and yesterday's decisions. That's what an AI agent without cross-session memory feels like. The most helpful assistants are the ones that learn and remember over time.

Without cross-session memory, every conversation starts from zero. A user who spent thirty minutes teaching an agent returns the next day to a blank slate. This doesn't only waste the user's time. It limits the value an agent can provide.

Cross-session memory fixes this by adding a persistence layer between sessions. "Persistence" means saving data so it survives after the program stops running. When a session ends, the agent saves its relevant state. This includes extracted facts, user preferences, conversation summaries, and task context. The system serializes (converts to a storable format) this state and writes it to a durable backend. When a new session begins, the system identifies the returning user and loads their memory snapshot. It initializes the agent with that context before the first turn. The result is an agent that "remembers" across days, weeks, and months.

The engineering challenges are practical but important. You need to choose a serialization format. JSON is human-readable. Pickle handles complex Python objects. You need to pick a storage backend. Redis is fast. SQLite is straightforward to set up. S3 scales well. You also need to handle the cold-start case when no prior memory exists. And you must decide how much prior context to load when stored history exceeds the context window.

**In this notebook you'll build:**
1. A `SessionState` data class for serializable agent state.
2. A `StorageBackend` with a SQLite implementation.
3. A `CrossSessionManager` for session lifecycle (save, load, cold start).
4. A `CrossSessionAgent` that uses persisted memory in the conversation loop.
5. A working demo across multiple sessions with the same user.

## Key Concepts

- **Session serialization**: Converting the agent's in-memory state into a storable format. That state includes conversation history, extracted facts, user preferences, and task context. JSON is human-readable and easy to inspect. Pickle handles arbitrary Python objects. Protobuf offers compact binary encoding with schema evolution (the ability to change your data format without breaking old data).

- **State persistence backends**: The durable store where serialized state lives between sessions. Redis offers sub-millisecond reads for low-latency resumption. SQLite provides zero-dependency local persistence. S3/GCS scale to millions of users with high durability. PostgreSQL supports complex queries over stored state.

- **Session resumption**: Detecting a returning user and loading their stored memory snapshot. The system hydrates (restores) the agent's internal state before the first conversational turn. This must be fast enough to avoid noticeable delay at session start.

- **User identification**: Mapping incoming requests to a persistent user identity (user ID, API key, session token, or authentication claims). The agent must retrieve the correct memory partition. Strict isolation prevents memory from leaking between users.

- **Memory loading strategies**: When stored history exceeds the context window, the system must choose what to load. Options include: full history, last-N turns, summary-only, or relevance-ranked retrieval. Relevance-ranked means querying stored memories for what's most relevant to the current conversation.

- **Cold start handling**: Initializing agent state when no prior session exists for a user. This includes setting default preferences and establishing baseline context.

## Architecture

Cross-session memory operates at the boundaries of sessions. It serializes state when a session ends and deserializes it when a new session starts. A durable storage backend bridges the gap.

<p align="center">
  <img src="../../images/diagrams/21_cross_session_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph "Session End"
        AS[Agent State] --> SER[Serializer<br/>JSON / pickle]
    end

    SER --> SB[(Storage Backend<br/>Redis / SQLite / S3)]

    subgraph "New Session"
        REQ[Incoming Request] --> UID[User<br/>Identifier]
        UID -->|user_id| ML[Memory<br/>Loader]
        SB -->|stored state| ML
        ML --> LS{Loading<br/>Strategy}
        LS -->|full| CTX1[Full History]
        LS -->|partial| CTX2[Last-N Turns]
        LS -->|summary| CTX3[Summary Only]
        CTX1 --> AC[Agent Context]
        CTX2 --> AC
        CTX3 --> AC
    end

    subgraph "Cold Start"
        UID -->|no prior state| CS[Cold Start<br/>Handler]
        CS -->|defaults +<br/>onboarding| AC
    end

    style SB fill:#4a9eff,color:#fff
    style UID fill:#845ef7,color:#fff
    style LS fill:#ffa94d,color:#fff
    style CS fill:#ff6b6b,color:#fff
```

</details>

**Data flow:** At session end, the Agent State passes through a Serializer (JSON or pickle). The serialized state goes to the Storage Backend, keyed by user ID. When a new session begins, the User Identifier extracts the user's identity from the incoming request. The Memory Loader queries the Storage Backend for that user's stored state. The Loading Strategy decides how much to load based on history size and context window limits. If no prior state exists, the Cold Start Handler initializes defaults. The loaded state then gets injected into the Agent Context.

## Setup

Install dependencies and configure API access.

In [ ]:
%pip install -q openai python-dotenv

Import the OpenAI SDK and standard library helpers. The API key loads from a `.env` file.

In [ ]:
import os
import json
import sqlite3
from datetime import datetime
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod
from typing import Optional

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

## Implementation

We'll build cross-session memory in four parts:

1. A `SessionState` data class that holds everything the agent needs to remember.
2. A `StorageBackend` that persists state to SQLite.
3. A `CrossSessionManager` that handles saving, loading, and cold starts.
4. A `CrossSessionAgent` that wires memory into the conversation loop.

### Session State

The first step is defining *what* to save. We capture five things: a conversation summary, extracted facts, user preferences, recent messages, and session metadata.

In [ ]:
@dataclass
class SessionState:
    """Snapshot of agent state that persists between sessions."""

    user_id: str
    conversation_summary: str
    extracted_facts: list[str]
    user_preferences: dict
    last_n_messages: list[dict]
    session_count: int = 0
    last_active: str = ""

    def to_dict(self) -> dict:
        """Convert to a JSON-serializable dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "SessionState":
        """Reconstruct a SessionState from a dictionary."""
        return cls(**data)

### Storage Backend

Think of a storage backend like a filing cabinet. Each drawer is labeled with a user ID. Inside each drawer sits a folder containing that user's session state. The backend's job is to file and retrieve these folders.

We define an abstract interface so you can swap backends without changing the rest of the code. Then we build a concrete backend using SQLite, a lightweight database stored in a single file.

In [ ]:
class StorageBackend(ABC):
    """Interface for session state persistence."""

    @abstractmethod
    def save(self, user_id: str, state: SessionState) -> None: ...

    @abstractmethod
    def load(self, user_id: str) -> Optional[SessionState]: ...

    @abstractmethod
    def delete(self, user_id: str) -> None: ...

    @abstractmethod
    def list_users(self) -> list[str]: ...


class SQLiteBackend(StorageBackend):
    """Persist session state in a SQLite database."""

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS session_state (
                user_id    TEXT PRIMARY KEY,
                state_json TEXT NOT NULL,
                updated_at TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def save(self, user_id: str, state: SessionState) -> None:
        self.conn.execute(
            """INSERT OR REPLACE INTO session_state
               (user_id, state_json, updated_at) VALUES (?, ?, ?)""",
            (user_id, json.dumps(state.to_dict()), datetime.now().isoformat()),
        )
        self.conn.commit()

    def load(self, user_id: str) -> Optional[SessionState]:
        row = self.conn.execute(
            "SELECT state_json FROM session_state WHERE user_id = ?",
            (user_id,),
        ).fetchone()
        if row is None:
            return None
        return SessionState.from_dict(json.loads(row[0]))

    def delete(self, user_id: str) -> None:
        self.conn.execute(
            "DELETE FROM session_state WHERE user_id = ?", (user_id,),
        )
        self.conn.commit()

    def list_users(self) -> list[str]:
        rows = self.conn.execute("SELECT user_id FROM session_state").fetchall()
        return [r[0] for r in rows]

### LLM-Powered Fact Extraction and Summarization

At the end of each session, we ask the LLM to pull out key facts about the user. We also compress the conversation into a short summary. These compact representations carry the most important information into the next session without replaying every message.

In [ ]:
def extract_facts(messages: list[dict], llm_client: OpenAI) -> list[str]:
    """Ask the LLM to extract key user facts from conversation messages."""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract key facts about the user from this conversation. "
                    "Return each fact on its own line, prefixed with '- '. "
                    "Focus on: name, role, projects, preferences, and goals."
                ),
            },
            {"role": "user", "content": transcript},
        ],
        max_tokens=300,
    )
    raw = response.choices[0].message.content
    return [
        line.strip("- ").strip()
        for line in raw.strip().split("\n")
        if line.strip()
    ]


def summarize_conversation(messages: list[dict], llm_client: OpenAI) -> str:
    """Compress conversation history into a short summary."""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Summarize this conversation in 2-3 sentences. "
                    "Capture the main topics and any decisions made."
                ),
            },
            {"role": "user", "content": transcript},
        ],
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()

### Cross-Session Manager

The manager sits between the agent and the storage backend. It handles three responsibilities:

1. **Resuming** a session: load stored state, increment the session counter, apply the loading strategy.
2. **Ending** a session: serialize the current state and write it to storage.
3. **Cold starts**: create default state when a user has no history.

In [ ]:
class CrossSessionManager:
    """Orchestrates session persistence, resumption, and cold starts."""

    def __init__(
        self,
        backend: StorageBackend,
        loading_strategy: str = "full",
        max_messages: int = 20,
    ):
        self.backend = backend
        self.loading_strategy = loading_strategy
        self.max_messages = max_messages  # used by the "last_n" strategy

    def resume_session(self, user_id: str) -> SessionState:
        """Load prior state or cold-start a new user."""
        state = self.backend.load(user_id)
        if state is None:
            print(f"[Cold start] No prior state for '{user_id}'. Creating fresh state.")
            return self._cold_start(user_id)

        state.session_count += 1
        state.last_active = datetime.now().isoformat()
        loaded = self._apply_loading_strategy(state)
        print(
            f"[Resumed] User '{user_id}', session #{loaded.session_count}. "
            f"Loaded {len(loaded.last_n_messages)} messages, "
            f"{len(loaded.extracted_facts)} facts."
        )
        return loaded

    def end_session(self, user_id: str, state: SessionState) -> None:
        """Persist current state to the backend."""
        state.last_active = datetime.now().isoformat()
        self.backend.save(user_id, state)
        print(
            f"[Saved] User '{user_id}': "
            f"{len(state.last_n_messages)} messages, "
            f"{len(state.extracted_facts)} facts persisted."
        )

Next we define two helper methods on `CrossSessionManager`.
`_cold_start` creates default state for a first-time user.
`_apply_loading_strategy` trims the loaded state based on the chosen strategy (full, last-N, or summary-only).

In [ ]:
    def _cold_start(self, user_id: str) -> SessionState:
        """Initialize default state for a first-time user."""
        return SessionState(
            user_id=user_id,
            conversation_summary="",
            extracted_facts=[],
            user_preferences={},
            last_n_messages=[],
            session_count=1,
            last_active=datetime.now().isoformat(),
        )

    def _apply_loading_strategy(self, state: SessionState) -> SessionState:
        """Trim loaded state based on the chosen strategy."""
        if self.loading_strategy == "full":
            return state
        elif self.loading_strategy == "last_n":
            # Keep only the most recent messages
            state.last_n_messages = state.last_n_messages[-self.max_messages :]
            return state
        elif self.loading_strategy == "summary":
            # Drop messages entirely; rely on summary + facts
            state.last_n_messages = []
            return state
        return state

### Putting It Together: The Agent

The agent wires cross-session memory into the LLM conversation loop. On each turn it builds a system prompt from the stored summary, extracted facts, and session metadata. This gives the model context about the user before it reads the new message.

In [ ]:
class CrossSessionAgent:
    """Chat agent with memory that persists across sessions."""

    def __init__(self, manager: CrossSessionManager, model: str = "gpt-4o-mini"):
        self.manager = manager
        self.model = model
        self.client = OpenAI()
        self.state: Optional[SessionState] = None
        self.user_id: Optional[str] = None

    def start_session(self, user_id: str) -> None:
        """Resume or cold-start a session for the given user."""
        self.user_id = user_id
        self.state = self.manager.resume_session(user_id)

    def chat(self, user_input: str) -> str:
        """Send a message and get a response with cross-session context."""
        self.state.last_n_messages.append(
            {"role": "user", "content": user_input}
        )

        # Build a system prompt from persisted memory
        system_parts = [
            "You are a helpful assistant with memory across sessions.",
            "Keep your replies concise (2-3 sentences).",
        ]
        if self.state.conversation_summary:
            system_parts.append(
                f"\nPrevious conversation summary:\n{self.state.conversation_summary}"
            )
        if self.state.extracted_facts:
            facts_str = "\n".join(f"- {f}" for f in self.state.extracted_facts)
            system_parts.append(f"\nKnown facts about this user:\n{facts_str}")
        if self.state.session_count > 1:
            system_parts.append(
                f"\nThis is session #{self.state.session_count} with this user."
            )

        api_messages = [
            {"role": "system", "content": "\n".join(system_parts)},
        ] + self.state.last_n_messages

        response = self.client.chat.completions.create(
            model=self.model,
            messages=api_messages,
            max_tokens=300,
        )

        reply = response.choices[0].message.content
        self.state.last_n_messages.append(
            {"role": "assistant", "content": reply}
        )
        return reply

When a session ends, the agent extracts facts and summarizes the conversation.
It then persists this state through the manager.
This is the bridge between the live conversation and durable storage.

In [ ]:
    def end_session(self) -> None:
        """Extract facts, summarize, and persist state."""
        if self.state and len(self.state.last_n_messages) > 0:
            self.state.extracted_facts = extract_facts(
                self.state.last_n_messages, self.client
            )
            self.state.conversation_summary = summarize_conversation(
                self.state.last_n_messages, self.client
            )
        self.manager.end_session(self.user_id, self.state)

## Example Run

We'll simulate two sessions with the same user, then a cold start with a new user. This shows how cross-session memory creates continuity.

### Session 1: First Meeting

Alice chats with the agent for the first time. The manager triggers a cold start because no prior state exists. At the end, we save her session.

In [ ]:
backend = SQLiteBackend()  # in-memory SQLite for this demo
manager = CrossSessionManager(backend, loading_strategy="full")
agent = CrossSessionAgent(manager)

agent.start_session("alice_123")

print("=" * 50)
print("SESSION 1: First Meeting")
print("=" * 50, "\n")

session_1_messages = [
    "Hi! I'm Alice. I'm a data scientist at Acme Corp.",
    "I'm building a recommendation system with collaborative filtering.",
    "I prefer Python and use pandas for most of my data work.",
]

for msg in session_1_messages:
    print(f"User:  {msg}")
    reply = agent.chat(msg)
    print(f"Agent: {reply}\n")

agent.end_session()

### Session 2: The Agent Remembers

Alice returns later. The agent loads her stored summary and facts. It should recall her name, role, and project without her repeating them.

In [ ]:
agent_s2 = CrossSessionAgent(manager)
agent_s2.start_session("alice_123")

print("=" * 50)
print("SESSION 2: Alice Returns")
print("=" * 50, "\n")

session_2_messages = [
    "Hey, I'm back! What do you remember about me?",
    "Any suggestions for improving my recommendation system?",
]

for msg in session_2_messages:
    print(f"User:  {msg}")
    reply = agent_s2.chat(msg)
    print(f"Agent: {reply}\n")

agent_s2.end_session()

### Cold Start: New User

Bob arrives with no history. The manager creates default state and the agent starts fresh.

In [ ]:
agent_new = CrossSessionAgent(manager)
agent_new.start_session("bob_456")

print("=" * 50)
print("NEW USER: Cold Start")
print("=" * 50, "\n")

msg = "Hello! Can you help me with a machine learning project?"
print(f"User:  {msg}")
reply = agent_new.chat(msg)
print(f"Agent: {reply}\n")

agent_new.end_session()

### Inspecting Stored State

Let's look at what the backend saved for Alice after two sessions. You'll see the extracted facts, conversation summary, and session metadata.

In [ ]:
alice_state = backend.load("alice_123")

print("Stored State for alice_123")
print("-" * 40)
print(f"Session count: {alice_state.session_count}")
print(f"Last active:   {alice_state.last_active}")
print(f"\nExtracted facts:")
for fact in alice_state.extracted_facts:
    print(f"  - {fact}")
print(f"\nConversation summary:")
print(f"  {alice_state.conversation_summary}")
print(f"\nStored messages: {len(alice_state.last_n_messages)}")
print(f"All users in backend: {backend.list_users()}")

### Comparing Loading Strategies

When stored history grows large, you need to choose what to load. Here we compare three strategies using Alice's saved state:

- **full**: Load everything. Best for users with short histories.
- **last_n**: Keep only the most recent N messages. Trims old context.
- **summary**: Drop all messages. Rely on the summary and extracted facts only.

In [ ]:
print("Loading Strategy Comparison (alice_123)")
print("=" * 50, "\n")

for strategy in ["full", "last_n", "summary"]:
    test_manager = CrossSessionManager(
        backend, loading_strategy=strategy, max_messages=4
    )
    loaded = test_manager.resume_session("alice_123")
    print(f"Strategy: {strategy}")
    print(f"  Messages loaded:  {len(loaded.last_n_messages)}")
    print(f"  Facts available:  {len(loaded.extracted_facts)}")
    has_summary = "yes" if loaded.conversation_summary else "no"
    print(f"  Summary present:  {has_summary}")
    print()

Clean up the in-memory database.

In [ ]:
backend.conn.close()

## Tradeoffs

### When Cross-Session Memory Works Well

- **Personalized assistants.** Users who return regularly get a better experience. The agent remembers their name, preferences, and ongoing projects.
- **Long-running tasks.** A multi-day research project can pick up where it left off. The agent tracks what's been covered and what remains.
- **Reducing user effort.** Users don't need to repeat context. This saves time and reduces frustration.

### When It Breaks Down

- **Privacy and data retention.** Storing user data across sessions raises compliance questions (GDPR, CCPA). You need clear policies for how long data persists and how users can delete it.
- **Stale memories.** Facts extracted months ago may no longer be true. A user might change jobs or switch projects. Without a way to expire old facts, the agent can make wrong assumptions.
- **Storage costs at scale.** With millions of users, the storage backend becomes a real infrastructure concern. You need to plan for database sizing, backups, and access patterns.
- **Context window limits.** Even with loading strategies, accumulated state from many sessions can exceed what fits in a single prompt. You'll need to combine this technique with summarization or retrieval-based approaches.

## Further Reading

- Packer, C., et al. (2023). ["MemGPT: Towards LLMs as Operating Systems."](https://arxiv.org/abs/2310.08560) Introduces virtual memory paging for LLMs. The agent works with a bounded context window but pages in data from persistent external storage.

- [LangChain Memory Documentation](https://python.langchain.com/docs/modules/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Covers conversation memory types (buffer, summary, entity) with built-in persistence to files and databases.

- [Letta (MemGPT) Persistence Model](https://docs.letta.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Documents Letta's approach to durable agent state across sessions, including memory hierarchy and backend architecture.

- [SQLite: When to Use](https://www.sqlite.org/whentouse.html?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Guidance on when SQLite is the right database choice. Relevant because many agent prototypes use SQLite for local state persistence.

- [Redis Persistence Documentation](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/): Explains Redis durability options (RDB snapshots, AOF logs). Useful when you need sub-millisecond session resumption at production scale.

---

*\u2190 Previous: [20: Memory Retrieval Patterns](../20_memory_retrieval_patterns/) \u00b7 Next: [22: Multi-Agent Shared Memory](../22_multi_agent_shared_memory/) \u2192*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: State versioning
Add a `version` integer to `SessionState`. Each time `CrossSessionManager` saves state, increment the version and keep the previous version in a `history` table. Implement a `rollback(version)` method that restores a previous state. Test by saving 3 versions and rolling back to the first.

### Challenge 2: Storage growth tracking
After every 5 turns in a conversation, serialize the `SessionState` to JSON and record its byte size. Run a 30-turn conversation and plot turn number vs. state size. Identify which component (messages, facts, or preferences) grows fastest.

### Challenge 3: Multi-user session manager
Extend `CrossSessionManager` to accept a `user_id` parameter. Store each user's state in a separate SQLite row (or file). Create two users, run 5 turns each, save, and then resume both. Verify that their states are isolated. This complements the multi-agent patterns in 22 Multi-Agent Shared Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--21-cross-session-memory--cross-session-memory)